# Visual Breed Prediction – stima razza usando tutte le foto

Questo notebook applica la stima visiva della razza ai cani indicati come `Mixed Breed`.

Per ogni cane vengono considerate **tutte le foto disponibili**.  
Per ogni foto il modello ResNet50 produce una probabilità sulle classi canine di ImageNet.  
Le probabilità vengono poi aggregate facendo la media tra tutte le foto dello stesso cane.

Il risultato finale viene salvato in un CSV con:

- `visual_breed_1`
- `visual_breed_1_score`
- `visual_breed_2`
- `visual_breed_2_score`
- `visual_breed_3`
- `visual_breed_3_score`

Queste colonne rappresentano una **stima visiva**, non una razza certa.


## 1. Import delle librerie

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

import torch
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image

from IPython.display import display
from tqdm import tqdm

## 2. Caricamento dataset e immagini

In [ ]:
dogs = pd.read_csv("../data/processed/dogs_clean.csv")

images_dir = Path("../data/raw/train_images")
results_dir = Path("../data/results")
results_dir.mkdir(parents=True, exist_ok=True)

print("Dataset:", dogs.shape)
print("Cartella immagini esiste:", images_dir.exists())

mixed_dogs = dogs[dogs["breed1_label"] == "Mixed Breed"].copy()

print("Numero cani Mixed Breed:", len(mixed_dogs))

mixed_dogs[["PetID", "Name", "breed1_label", "PhotoAmt"]].head()

## 3. Caricamento modello ResNet50

Viene usata ResNet50 pre-addestrata su ImageNet.

ImageNet contiene molte classi di razze canine, quindi il modello può essere usato per ottenere una stima visiva della razza.


In [ ]:
weights = ResNet50_Weights.DEFAULT

model = resnet50(weights=weights)
model.eval()

preprocess = weights.transforms()
categories = weights.meta["categories"]

print("Numero classi ImageNet:", len(categories))

## 4. Selezione delle sole classi canine

ImageNet contiene molte classi non canine.  
Per evitare predizioni non pertinenti, vengono mantenute solo le classi che rappresentano razze o categorie canine.


In [ ]:
dog_keywords = [
    "terrier",
    "retriever",
    "spaniel",
    "shepherd",
    "hound",
    "poodle",
    "husky",
    "beagle",
    "chihuahua",
    "doberman",
    "rottweiler",
    "collie",
    "corgi",
    "bulldog",
    "mastiff",
    "pinscher",
    "setter",
    "dane",
    "wolfhound",
    "malamute",
    "samoyed",
    "schipperke",
    "affenpinscher"
]

dog_class_indices = [
    i for i, cls in enumerate(categories)
    if any(word in cls.lower() for word in dog_keywords)
]

dog_classes = [categories[i] for i in dog_class_indices]

print("Numero classi canine trovate:", len(dog_classes))
dog_classes

## 5. Funzioni per trovare immagini e predire razze

Per ogni cane vengono cercate tutte le immagini con nome:

```text
PetID-1.jpg
PetID-2.jpg
PetID-3.jpg
...
```

Poi si calcolano le probabilità ResNet50 per ciascuna immagine e si aggregano le probabilità canine facendo la media.


In [ ]:
def get_pet_images(pet_id):
    images = sorted(images_dir.glob(f"{pet_id}-*.jpg"))
    return images


def predict_dog_probabilities_for_image(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception:
        return None

    input_tensor = preprocess(image).unsqueeze(0)

    with torch.no_grad():
        outputs = model(input_tensor)

    probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    dog_probabilities = probabilities[dog_class_indices].numpy()

    return dog_probabilities


def top_dog_predictions_from_probs(dog_probabilities, top_k=3):
    top_indices = np.argsort(dog_probabilities)[::-1][:top_k]

    predictions = []

    for idx in top_indices:
        predictions.append({
            "breed": dog_classes[idx],
            "score": float(dog_probabilities[idx])
        })

    return predictions


def predict_visual_breeds_for_pet(pet_id, top_k=3):
    image_paths = get_pet_images(pet_id)

    if len(image_paths) == 0:
        return {
            "num_images_used": 0,
            "visual_breed_1": None,
            "visual_breed_1_score": 0.0,
            "visual_breed_2": None,
            "visual_breed_2_score": 0.0,
            "visual_breed_3": None,
            "visual_breed_3_score": 0.0
        }

    all_dog_probs = []

    for image_path in image_paths:
        dog_probs = predict_dog_probabilities_for_image(image_path)

        if dog_probs is not None:
            all_dog_probs.append(dog_probs)

    if len(all_dog_probs) == 0:
        return {
            "num_images_used": 0,
            "visual_breed_1": None,
            "visual_breed_1_score": 0.0,
            "visual_breed_2": None,
            "visual_breed_2_score": 0.0,
            "visual_breed_3": None,
            "visual_breed_3_score": 0.0
        }

    mean_probs = np.mean(all_dog_probs, axis=0)
    top_predictions = top_dog_predictions_from_probs(mean_probs, top_k=top_k)

    result = {
        "num_images_used": len(all_dog_probs)
    }

    for rank, pred in enumerate(top_predictions, start=1):
        result[f"visual_breed_{rank}"] = pred["breed"]
        result[f"visual_breed_{rank}_score"] = pred["score"]

    return result

## 6. Esempio dettagliato su un cane

Questa sezione mostra un esempio completo su un singolo cane.

Per il cane scelto vengono visualizzate:

- tutte le sue foto disponibili;
- le top 3 razze canine stimate per ogni singola foto;
- il risultato aggregato finale, ottenuto facendo la media delle probabilità sulle foto.

Questo serve a capire perché usare tutte le immagini può essere più robusto rispetto a usare una sola foto.


In [ ]:
example_pet_id = "3422e4906"  # Brisco

example_row = dogs[dogs["PetID"] == example_pet_id].iloc[0]
example_images = get_pet_images(example_pet_id)

print("PetID:", example_pet_id)
print("Nome:", example_row["Name"])
print("Razza originale:", example_row["breed1_label"], "/", example_row["breed2_label"])
print("Numero immagini trovate:", len(example_images))

In [ ]:
photo_level_results = []
all_photo_probs = []

for image_path in example_images:
    print("=" * 80)
    print("Foto:", image_path.name)

    image = Image.open(image_path).convert("RGB")
    display(image)

    dog_probs = predict_dog_probabilities_for_image(image_path)

    if dog_probs is None:
        print("Errore nella lettura dell'immagine.")
        continue

    all_photo_probs.append(dog_probs)

    top_predictions = top_dog_predictions_from_probs(dog_probs, top_k=3)

    for rank, pred in enumerate(top_predictions, start=1):
        print(f'{rank}. {pred["breed"]} - {pred["score"] * 100:.2f}%')

        photo_level_results.append({
            "PetID": example_pet_id,
            "image": image_path.name,
            "rank": rank,
            "visual_breed": pred["breed"],
            "score": pred["score"]
        })

photo_level_df = pd.DataFrame(photo_level_results)
photo_level_df

In [ ]:
mean_probs = np.mean(all_photo_probs, axis=0)

aggregated_predictions = top_dog_predictions_from_probs(mean_probs, top_k=3)

print("RISULTATO AGGREGATO SU TUTTE LE FOTO")
print("Numero foto usate:", len(all_photo_probs))
print()

for rank, pred in enumerate(aggregated_predictions, start=1):
    print(f'{rank}. {pred["breed"]} - {pred["score"] * 100:.2f}%')

aggregated_example_df = pd.DataFrame([
    {
        "rank": rank,
        "visual_breed": pred["breed"],
        "average_score": pred["score"]
    }
    for rank, pred in enumerate(aggregated_predictions, start=1)
])

aggregated_example_df

## 7. Test su pochi cani

Prima di applicare il modello a tutti i `Mixed Breed`, viene eseguita una prova sui primi 10 cani.

Se i risultati sembrano plausibili, si può eseguire la cella successiva su tutti i cani.


In [ ]:
test_petids = mixed_dogs["PetID"].head(10).tolist()

test_results = []

for pet_id in tqdm(test_petids):
    row = mixed_dogs[mixed_dogs["PetID"] == pet_id].iloc[0]
    prediction = predict_visual_breeds_for_pet(pet_id)

    test_results.append({
        "PetID": pet_id,
        "Name": row["Name"],
        "PhotoAmt": row["PhotoAmt"],
        **prediction
    })

test_df = pd.DataFrame(test_results)

test_df

## 8. Predizione su tutti i cani Mixed Breed

Questa cella può richiedere tempo perché analizza molte immagini.

Nel tuo dataset ci sono circa 5923 cani `Mixed Breed` e oltre 58.000 immagini totali.

Se vuoi prima fare una prova più piccola, cambia:

```python
mixed_to_process = mixed_dogs
```

in:

```python
mixed_to_process = mixed_dogs.head(100)
```


In [ ]:
mixed_to_process = mixed_dogs

visual_results = []

for _, row in tqdm(
    mixed_to_process.iterrows(),
    total=len(mixed_to_process)
):
    pet_id = row["PetID"]

    prediction = predict_visual_breeds_for_pet(pet_id)

    visual_results.append({
        "PetID": pet_id,
        "Name": row["Name"],
        "breed1_label": row["breed1_label"],
        "breed2_label": row["breed2_label"],
        "PhotoAmt": row["PhotoAmt"],
        **prediction
    })

visual_breed_predictions = pd.DataFrame(visual_results)

visual_breed_predictions.head()

## 9. Salvataggio CSV

Il file viene salvato in:

```text
data/results/visual_breed_predictions_mixed_breed.csv
```

Questo CSV potrà essere usato in un notebook successivo per integrare le razze visive nel sistema di matching.


In [ ]:
output_file = results_dir / "visual_breed_predictions_mixed_breed.csv"

visual_breed_predictions.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print("File salvato in:")
print(output_file)

print("Numero righe salvate:")
print(len(visual_breed_predictions))

## 10. Controllo finale

Mostra i primi risultati ordinati per score della prima razza visiva.


In [ ]:
visual_breed_predictions.sort_values(
    by="visual_breed_1_score",
    ascending=False
).head(20)

## 11. Conclusioni

Questo notebook produce una stima visiva delle razze più probabili per i cani `Mixed Breed`.

La stima considera tutte le foto disponibili per ciascun cane e aggrega i risultati tramite media delle probabilità.

Queste informazioni non sostituiscono la razza reale, ma possono essere usate come informazione ausiliaria nel sistema di matching cane–famiglia.

Nel prossimo passo si potrà integrare il file CSV prodotto nel ranking finale, assegnando un punteggio parziale quando una razza preferita dalla famiglia corrisponde a una delle razze visive stimate.
